<!-- NB_DOC_INTRO_V1 -->
# DL — Mathematiques du Deep Learning (NumPy from-scratch)

> 📚 **Doc thematique** : [docs/04_DL.md](docs/04_DL.md) (Deep Learning)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**LE notebook pour comprendre vraiment** comment marche un reseau de neurones : tout est code en **NumPy pur**, aucun framework. Forward pass, backpropagation, gradient descent — implementes ligne par ligne pour un MLP a N couches.

Apres avoir lu ce notebook, PyTorch / TensorFlow deviendront limpides : ce sont juste des optimiseurs autograd qui font ce qu'on fait ici a la main.

## Plan

1. Setup et notations
2. Activations (sigmoid, ReLU, tanh, softmax) + leurs derivees
3. Loss functions (MSE, BCE, cross-entropy)
4. Forward propagation (1 couche, puis N couches)
5. Backpropagation (chain rule)
6. Gradient descent (SGD, mini-batch)
7. Optimisations : momentum, Adam (concepts)
8. Demo : MLP 2-couches sur make_moons
9. Visualisation de la convergence + decision boundary
10. Pieges et anti-patterns
11. References


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
SEED = 42
np.random.seed(SEED)

## 1. Notations standard

Pour un reseau a `L` couches :
- `X` : input shape `(n_samples, n_features)`
- `W^[l]` : poids couche `l`, shape `(n_in, n_out)`
- `b^[l]` : biais couche `l`, shape `(1, n_out)`
- `Z^[l] = A^[l-1] @ W^[l] + b^[l]` : pre-activation
- `A^[l] = g(Z^[l])` : activation (apres g = ReLU, sigmoid, ...)
- `A^[0] = X` (input layer)
- `y_hat = A^[L]` (output)


## 2. Activations + derivees

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))   # clip pour eviter overflow

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def tanh(z):
    return np.tanh(z)

def tanh_deriv(z):
    return 1 - np.tanh(z)**2

# Demo visuelle
z = np.linspace(-5, 5, 100)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, fn, deriv) in zip(axes, [
    ("Sigmoid", sigmoid, sigmoid_deriv),
    ("ReLU", relu, relu_deriv),
    ("Tanh", tanh, tanh_deriv),
]):
    ax.plot(z, fn(z), label=name)
    ax.plot(z, deriv(z), label=f"{name}'", linestyle="--")
    ax.legend(); ax.grid(alpha=0.3); ax.set_title(name)
plt.tight_layout(); plt.show()

## 3. Forward propagation

`Z = A @ W + b`, puis `A_new = g(Z)`.

In [ ]:
def init_params(layer_dims, seed=SEED):
    """Init Xavier/He pour chaque couche."""
    np.random.seed(seed)
    params = {}
    for l in range(1, len(layer_dims)):
        # He init pour ReLU : std = sqrt(2/n_in)
        params[f"W{l}"] = np.random.randn(layer_dims[l-1], layer_dims[l]) * np.sqrt(2.0 / layer_dims[l-1])
        params[f"b{l}"] = np.zeros((1, layer_dims[l]))
    return params

def forward(X, params, activation="relu"):
    """Forward pass. Stocke caches pour backprop."""
    A = X
    caches = {"A0": X}
    L = len(params) // 2

    for l in range(1, L):
        Z = A @ params[f"W{l}"] + params[f"b{l}"]
        A = relu(Z) if activation == "relu" else tanh(Z)
        caches[f"Z{l}"] = Z
        caches[f"A{l}"] = A

    # Derniere couche : sigmoid (binaire) ou softmax (multi-classe)
    Z_L = A @ params[f"W{L}"] + params[f"b{L}"]
    A_L = sigmoid(Z_L)
    caches[f"Z{L}"] = Z_L
    caches[f"A{L}"] = A_L
    return A_L, caches

# Test
params = init_params([2, 4, 1])
X_test = np.random.randn(5, 2)
y_pred, _ = forward(X_test, params)
print(f"y_pred shape : {y_pred.shape}, values : {y_pred.ravel().round(3)}")

## 4. Loss (BCE) + cost

In [ ]:
def bce_loss(y_true, y_pred, eps=1e-15):
    """Binary cross-entropy. y_true et y_pred : shape (n_samples, 1)."""
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

y_true = np.array([[1], [0], [1], [0]])
y_pred = np.array([[0.9], [0.1], [0.8], [0.3]])
print(f"BCE loss : {bce_loss(y_true, y_pred):.4f}")

## 5. Backpropagation

Application de la **chain rule** : `dZ^[L] = A^[L] - y`, puis on remonte couche par couche.

In [ ]:
def backward(y_true, params, caches, activation="relu"):
    grads = {}
    L = len(params) // 2
    m = y_true.shape[0]   # batch size

    # Derniere couche (sigmoid + BCE → dZ = A - y, formule magique)
    dZ = caches[f"A{L}"] - y_true
    grads[f"dW{L}"] = caches[f"A{L-1}"].T @ dZ / m
    grads[f"db{L}"] = np.sum(dZ, axis=0, keepdims=True) / m

    for l in reversed(range(1, L)):
        dA = dZ @ params[f"W{l+1}"].T
        deriv = relu_deriv if activation == "relu" else tanh_deriv
        dZ = dA * deriv(caches[f"Z{l}"])
        grads[f"dW{l}"] = caches[f"A{l-1}"].T @ dZ / m
        grads[f"db{l}"] = np.sum(dZ, axis=0, keepdims=True) / m

    return grads

def update_params(params, grads, lr):
    """SGD step."""
    L = len(params) // 2
    for l in range(1, L + 1):
        params[f"W{l}"] -= lr * grads[f"dW{l}"]
        params[f"b{l}"] -= lr * grads[f"db{l}"]
    return params

print("Backward + update OK")

## 6. Gradient descent — boucle d'entrainement complete

In [ ]:
def train(X, y, layer_dims, n_iters=2000, lr=0.05, activation="relu", verbose=True):
    params = init_params(layer_dims)
    losses = []
    for i in range(n_iters):
        y_pred, caches = forward(X, params, activation)
        loss = bce_loss(y, y_pred)
        grads = backward(y, params, caches, activation)
        params = update_params(params, grads, lr)
        losses.append(loss)
        if verbose and (i % 400 == 0):
            acc = ((y_pred > 0.5) == y).mean()
            print(f"  Iter {i:4d}  loss={loss:.4f}  acc={acc:.4f}")
    return params, losses

# Demo sur make_moons
X, y = make_moons(n_samples=500, noise=0.2, random_state=SEED)
y = y.reshape(-1, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED)

params_trained, losses = train(X_tr, y_tr, [2, 16, 8, 1], n_iters=2000, lr=0.1, activation="relu")

# Eval
y_pred_te, _ = forward(X_te, params_trained)
acc_te = ((y_pred_te > 0.5) == y_te).mean()
print(f"\nTest accuracy : {acc_te:.4f}")

## 7. Visualisation — loss + decision boundary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(losses)
axes[0].set(xlabel="Iteration", ylabel="BCE loss", title="Training loss")
axes[0].grid(alpha=0.3)

# Decision boundary
xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 200),
                     np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 200))
Z, _ = forward(np.c_[xx.ravel(), yy.ravel()], params_trained)
Z = Z.reshape(xx.shape)

axes[1].contourf(xx, yy, Z, levels=[0, 0.5, 1], alpha=0.3, cmap="coolwarm")
axes[1].scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr.ravel(), cmap="coolwarm", edgecolors="k", s=30)
axes[1].set_title(f"Decision boundary (test acc = {acc_te:.2f})")
plt.tight_layout(); plt.show()

## 8. Optimiseurs avances (concepts)

| Optimizer | Update rule | Quand |
|---|---|---|
| **SGD** | `θ -= lr × ∇` | Baseline, lent |
| **SGD + momentum** | `v = β·v + (1-β)·∇ ; θ -= lr × v` | Accelere convergence (β ~ 0.9) |
| **Nesterov** | Momentum avec correction lookahead | Variante de momentum |
| **AdaGrad** | `lr` adaptatif par parametre | Trop agressif sur long training |
| **RMSProp** | AdaGrad avec EMA des grads² | Fixe AdaGrad |
| **Adam** | RMSProp + momentum | Standard 2024-2025 |
| **AdamW** | Adam + decoupled weight decay | Mieux pour Transformers |

Tu peux les coder all from-scratch — la formule Adam :
```
m = β1·m + (1-β1)·∇        # 1er moment (momentum)
v = β2·v + (1-β2)·∇²       # 2eme moment (variance)
m_hat = m / (1 - β1^t)      # bias correction
v_hat = v / (1 - β2^t)
θ -= lr × m_hat / (sqrt(v_hat) + ε)
```


## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Init poids a zero | Symetrie : tous les neurones identiques |
| LR trop grand | Loss explose ou oscille |
| LR trop petit | Convergence ultra lente |
| Sigmoid sur caches profondes | Vanishing gradient — preferer ReLU |
| ReLU avec init bad | Dead neurons (toujours 0) — utiliser He init (sqrt(2/n)) |
| Pas normaliser X | Convergence lente, instabilite |
| Forward sans cache | Backprop impossible (besoin de Z et A intermediaires) |
| Pas de mini-batch | Gradient bruyant, lent (batch=1) ou plat (batch=N) |
| Activation softmax sans temperature | Sortie quasi-binaire si bcp de classes |


## References

### Documentation
- *Deep Learning* (Goodfellow, Bengio, Courville) - gratuit : https://www.deeplearningbook.org/
- 3blue1brown Neural Networks (videos) : https://www.3blue1brown.com/topics/neural-networks
- Karpathy — *makemore* : https://github.com/karpathy/makemore
- Andrew Ng Coursera DL Specialization

### Voir aussi
- [DL_Tensorflow_Keras.ipynb](DL_Tensorflow_Keras.ipynb)
- [DL_PyTorch.ipynb](DL_PyTorch.ipynb)
- [DL_PyTorch_Lightning.ipynb](DL_PyTorch_Lightning.ipynb)
- [KAN (Kolmogorov-Arnold Networks).ipynb](KAN (Kolmogorov-Arnold Networks).ipynb)
